# 6장 실습 — RAPTOR 직접 짜기 (채점)

직접 짜는 셋 중 두 번째입니다. 채울 파일은 **`labs/ch06_raptor.py`** 입니다.
이 노트북이 아니라 그 파일을 고칩니다. 노트북은 채운 것을 바로 확인하는 용도입니다.

3장의 최단경로와 다른 점이 하나 있습니다.
버스는 아무 때나 출발하지 않고 시간표에 적힌 시각에만 출발합니다.
그래서 "가장 가까운 이웃"이 아니라 "지금 시각에 탈 수 있는 가장 이른 차"를 찾아야 합니다.

순서는 이렇습니다. 빈칸 세 개를 파일에 적힌 순서대로 채웁니다.

1. 손으로 답을 아는 정류장 5개짜리 시간표를 봅니다 (교재 6.5)
2. `earliest_trip` 을 채웁니다 — 정렬된 출발 시각에서 이분 탐색 (교재 6.2)
3. `TransitData.from_gtfs` 를 채웁니다 — GTFS 를 자료구조 네 개로 (교재 6.2 ~ 6.4)
4. `raptor` 를 채웁니다 — 라운드 탐색 (교재 6.5)
5. 하남 GTFS 로 옮기고 불변식을 확인합니다 (교재 6.7)
6. `check("ch06")` 으로 채점합니다

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect
from smartmob.viz import use_korean_font

use_korean_font()

import ch06_raptor as sol      # 여러분이 채우는 파일

INF = float("inf")

`autoreload` 를 켜 두었으므로 `labs/ch06_raptor.py` 를 저장하면 이 노트북을 다시 시작하지 않아도 반영됩니다.

## 1. 손으로 답을 아는 시간표 (교재 6.5)

정류장 다섯 개, 노선 두 개, 운행 세 개입니다. 교재 6.5절의 그림과 같습니다.

```
  A ──(1호선)──► B ──(1호선)──► C      1호선  08:00 A → 08:10 B → 08:20 C
                 │                            08:30 A → 08:40 B → 08:50 C
             도보 약 100m
                 │
                 D ──(2호선)──► E      2호선  08:15 D → 08:25 E
```

A 에서 8시에 출발하면 라운드가 이렇게 진행됩니다. 라운드 번호가 곧 탄 횟수입니다.

| 라운드 | 무엇을 하는가 | 새로 도달한 정류장 |
|---|---|---|
| 0 | 출발지 A 에 08:00 을 적습니다 | A 08:00 |
| 1 | A 를 지나는 1호선을 훑습니다. 08:00 차를 타고 B, C 에 내립니다. B 에서 걸어서 D 로 갑니다 | B 08:10, C 08:20, D 08:11 |
| 2 | B, C, D 를 지나는 노선을 훑습니다. D 에서 08:15 2호선을 타고 E 에 내립니다 | E 08:25 |
| 3 | E 를 지나는 노선을 훑어도 나아지는 곳이 없습니다 | 없음, 끝 |

C 는 환승 0회에 08:20, E 는 환승 1회에 08:25 입니다. 8시 5분에 출발하면 첫 차를 놓쳐 C 가 08:50 이 됩니다.
종이에 그려 놓고 확인할 수 있는 크기입니다. 여기서 막히면 어디가 틀렸는지 바로 보입니다.

이 시간표가 `toy_feed()` 에 GTFS 표 네 개로 들어 있습니다. 5장에서 본 하남 GTFS 와 컬럼이 같습니다.

In [ ]:
from smartmob.teaching.raptor import toy_feed

feed = toy_feed()
for name, table in feed.items():
    print(f"{name:12s} {len(table):>4}행")

feed["stop_times"]

`stop_times` 8행이 위 그림의 시각 전부입니다. 운행 `L1-1` 과 `L1-2` 는 정류장 순서가 A→B→C 로 같으므로 뒤에서 패턴 하나로 묶입니다.

## 2. 첫 번째 빈칸 — earliest_trip (교재 6.2)

패턴 하나를 훑다가 어떤 정류장에 08:03 에 도착했다고 합시다. 그 정류장에서 08:03 이후 가장 먼저 출발하는 운행이 몇 번째인지 찾아야 합니다.
운행이 출발 시각 순으로 정렬되어 있으면 전부 훑을 필요가 없습니다. `bisect_left` 를 씁니다.

`bisect_left(정렬된_리스트, 값)` 은 그 값이 들어갈 자리의 번호를 돌려줍니다. 같은 값이 이미 있으면 그 값의 앞자리입니다.
그 번호가 곧 "값 이상인 첫 원소의 번호"이므로, 출발 시각 리스트에 걸면 "그 시각 이후 첫 운행의 번호"가 됩니다.
번호가 리스트 길이와 같으면 그 시각 이후에 출발하는 운행이 없다는 뜻입니다.

In [ ]:
from bisect import bisect_left

deps = [8 * 3600, 8 * 3600 + 30 * 60]          # 1호선이 A 를 떠나는 시각: 08:00, 08:30

for text, secs in [("07:50", 7 * 3600 + 50 * 60), ("08:00", 8 * 3600),
                   ("08:03", 8 * 3600 + 180), ("09:00", 9 * 3600)]:
    i = bisect_left(deps, secs)                 # secs 이상인 첫 원소의 번호
    found = f"{i}번 운행" if i < len(deps) else "없음 (리스트 끝)"
    print(f"{text} 이후 첫 운행 → bisect_left = {i} → {found}")

08:00 정각은 0번 운행을 탈 수 있고(같은 값이면 앞자리), 08:03 이면 1번, 09:00 이면 없음입니다.

`Pattern.earliest_trip(position, not_before)` 이 하는 일이 이 두 줄입니다.
`build_index()` 가 정류장 위치마다 출발 시각 열 `_dep_by_pos[position]` 을 만들어 둡니다.
그 열에 `bisect_left` 를 걸고, 번호가 끝이면 `None` 을 돌려줍니다.
`from_gtfs` 를 채우기 전에도 `Pattern` 을 손으로 만들어 확인할 수 있습니다. 시각은 자정부터의 초입니다.

In [ ]:
banner("earliest_trip 확인 (1호선 패턴을 손으로 만듦)")
line1 = sol.Pattern(
    name="1호선", route_type=1, stops=[0, 1, 2],
    arrivals=[[28800, 29400, 30000], [30600, 31200, 31800]],      # [운행][위치] 08:00, 08:10, 08:20 / 08:30, ...
    departures=[[28800, 29400, 30000], [30600, 31200, 31800]],
)
line1.build_index()
try:
    expect("A 에서 08:03 이후 첫 운행", line1.earliest_trip(0, 8 * 3600 + 180), 1)
    expect("A 에서 08:00 정각 첫 운행", line1.earliest_trip(0, 8 * 3600), 0)
    expect("B 에서 08:41 이후 운행이 없는가", line1.earliest_trip(1, 8 * 3600 + 41 * 60) is None, True)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)

## 3. 두 번째 빈칸 — from_gtfs (교재 6.2 ~ 6.4)

GTFS 표를 RAPTOR 가 쓰는 자료구조 네 개로 바꿉니다. 파일의 docstring 에 적힌 여섯 단계가 교재 6.2 ~ 6.4절 순서 그대로입니다.

1. `stops` 로 정류장 목록과 `stop_id → 인덱스` 사전 `index_of` 를 만듭니다
2. `stop_times` 를 `trip_id`, `stop_sequence` 로 정렬하고 시각을 `parse_gtfs_time` 으로 초로 바꿉니다
3. 운행을 정류장 순서가 같은 것끼리 묶어 **패턴** 을 만듭니다. 묶는 열쇠는 `(route_id, 정류장 인덱스 튜플)` 입니다
4. 각 패턴의 운행을 첫 정류장 출발 시각 순으로 정렬하고 `build_index()` 를 부릅니다
5. 정류장 → `[(패턴 번호, 위치), ...]` 역색인 `routes_by_stop` 을 만듭니다
6. 가까운 정류장 사이를 도보 환승 `transfers` 로 잇습니다

3번에서 `defaultdict(list)` 를 쓰면 처음 보는 열쇠에도 `append` 를 바로 할 수 있습니다.
6번에서 `cKDTree.query_ball_point(점, 반지름)` 은 반지름 안의 점 번호를 돌려줍니다. 4,203개 정류장을 전부 짝지으면 880만 쌍이라 이것으로 후보를 줄입니다.

같은 노선이라도 정류장 순서가 다르면 다른 패턴입니다. 작은 시간표에서는 패턴이 2개 나와야 합니다.

In [ ]:
def at(data, stop_id):
    """정류장 id → 인덱스. `index_of` 를 아직 안 만들었어도 동작합니다."""
    index = getattr(data, "index_of", None)
    if index and stop_id in index:
        return index[stop_id]
    return list(data.stop_ids).index(stop_id)


banner("작은 시간표로 자료구조 만들기")
try:
    toy = sol.TransitData.from_gtfs(feed)
    expect("정류장 수", len(toy.stop_ids), 5)
    expect("패턴 수", len(toy.patterns), 2)
    for i, p in enumerate(toy.patterns):
        print(f"    패턴 {i}: {p.name}  정류장 {[toy.stop_ids[s] for s in p.stops]}"
              f"  운행 {len(p.departures)}회  첫 출발 {[d[0] // 60 % 60 for d in p.departures]}분")
    b, d = at(toy, "B"), at(toy, "D")
    walk = dict(toy.transfers[b]).get(d)
    expect("B → D 도보 환승이 있는가", walk is not None, True)
    print(f"    B → D 도보 {walk}초" if walk else "    B → D 환승이 없습니다. 6번 단계를 봅니다")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

1호선 패턴의 운행이 `[0, 30]` 분 순서로 정렬되어 있어야 `earliest_trip` 이 맞게 동작합니다.
B 와 D 는 직선 100m 이므로 1.35 를 곱해 초속 1.2m 로 나누면 110초쯤이 나옵니다.

## 4. 세 번째 빈칸 — raptor (교재 6.5)

`raptor(data, origins, departure_secs)` 는 리스트 `best` 하나를 돌려줍니다. `best[i]` 가 정류장 i 의 가장 이른 도착시각(초)이고, 못 가면 `INF` 입니다.
교재 6.5절은 이것을 네 함수로 나눠 보여 줍니다. `collect_patterns`, `scan_pattern`, `walk_transfers`, `raptor_core` 입니다.
파일에서는 함수 하나에 다 적어도 됩니다.

들고 다니는 상태는 셋입니다.

- `best[정류장]` — 지금까지 알아낸 가장 이른 도착시각
- `rounds[k][정류장]` — k 번 타고 도달했을 때의 도착시각
- `marked` — 직전 라운드에서 개선된 정류장 집합

라운드 0 은 출발 정류장마다 `departure_secs + 도보시간` 을 적는 것입니다. 라운드 k 는 네 단계입니다.

1. `marked` 의 정류장을 지나는 패턴을 모읍니다. 같은 패턴이 여러 정류장에서 걸리면 가장 앞 위치 하나만 남깁니다
2. 패턴을 그 위치부터 끝까지 훑습니다. 정류장마다 **먼저 내려 보고**, 그다음 더 이른 차로 갈아탈 수 있는지 봅니다
3. 이번 라운드에 도달한 정류장에서 걸어갈 수 있는 곳을 채웁니다
4. 개선된 정류장이 없으면 끝냅니다

2번에서 갈아타는 판단에는 **직전 라운드** 의 도착시각을 씁니다. 이번 라운드 값을 쓰면 한 라운드에 여러 번 갈아타서 라운드 번호가 환승 횟수가 아니게 됩니다.
1절의 표에 있는 네 경우를 확인합니다.

In [ ]:
def hhmm(seconds):
    if seconds == INF:
        return "못 감"
    return f"{int(seconds) // 3600:02d}:{int(seconds) % 3600 // 60:02d}"


banner("작은 시간표 탐색")
try:
    toy = sol.TransitData.from_gtfs(feed)
    a, c, e = at(toy, "A"), at(toy, "C"), at(toy, "E")

    best = sol.raptor(toy, [(a, 0)], 8 * 3600)              # (정류장, 접근 도보 초)
    expect("A 08:00 출발 → C 도착", hhmm(best[c]), "08:20")
    expect("A 08:00 출발 → E 도착", hhmm(best[e]), "08:25")

    late = sol.raptor(toy, [(a, 0)], 8 * 3600 + 5 * 60)
    expect("A 08:05 출발 → C 도착", hhmm(late[c]), "08:50")

    night = sol.raptor(toy, [(a, 0)], 23 * 3600)
    expect("A 23:00 출발 → C 도착", hhmm(night[c]), "못 감")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

네 줄이 전부 통과하면 알고리즘이 맞는 것입니다. E 가 `못 감` 이면 3번 단계(도보 환승)나 `prev` 를 쓰는 자리를 봅니다.

라운드별로도 찍어 봅니다. `max_rounds` 를 0, 1, 2, 3 으로 두고 돌리면 k 라운드까지 도달한 정류장이 나옵니다.
1절의 표와 같은 정류장이 같은 라운드에 나와야 합니다.

In [ ]:
banner("라운드별 도달 정류장 (교재 6.5절 표와 비교)")
try:
    toy = sol.TransitData.from_gtfs(feed)
    a = at(toy, "A")
    seen = set()
    for k in range(4):
        best_k = sol.raptor(toy, [(a, 0)], 8 * 3600, max_rounds=k)
        new = {toy.stop_ids[i]: hhmm(t) for i, t in enumerate(best_k) if t < INF and i not in seen}
        seen |= {i for i, t in enumerate(best_k) if t < INF}
        print(f"라운드 {k}: 새로 도달 {new if new else '없음'}")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

라운드 1 에 B 08:10, C 08:20, D 08:11 이 나오고 라운드 2 에 E 08:25 가 나와야 합니다. 라운드 3 은 없음입니다.
E 가 라운드 1 에 나오면 갈아타는 판단에 이번 라운드 값을 쓴 것입니다.

## 5. 실제 하남 GTFS (교재 6.5, 6.7)

정류장이 5개에서 4,203개로, 운행이 3개에서 8,923개로 늘어납니다. 알고리즘은 그대로입니다.

In [ ]:
from smartmob.data import load_gtfs

hanam = load_gtfs("hanam")
print({k: len(v) for k, v in hanam.items()})

`from_gtfs` 는 시각표 65만 행을 초로 바꾸고 묶으므로 몇 초 걸립니다. 탐색은 그보다 훨씬 빠릅니다.
`access_stops` 는 좌표에서 걸어갈 수 있는 정류장과 도보 초를 돌려줍니다. 파일에 만들어 둔 함수입니다.

In [ ]:
import time

banner("하남 GTFS")
try:
    t0 = time.perf_counter()
    data = sol.TransitData.from_gtfs(hanam)
    n_transfers = sum(len(t) for t in data.transfers)
    print(f"[v] 자료구조 {time.perf_counter() - t0:.1f}초, 정류장 {len(data.stop_ids):,}개, "
          f"패턴 {len(data.patterns):,}개, 도보 환승 {n_transfers:,}쌍")

    origins = data.access_stops(37.5393, 127.2148)     # 하남시청에서 걸어갈 수 있는 정류장
    print(f"    출발 후보 정류장 {len(origins)}곳, 가장 가까운 곳까지 {origins[0][1]}초")

    t0 = time.perf_counter()
    best = sol.raptor(data, origins, 8 * 3600)
    reached = sum(1 for t in best if t < INF)
    print(f"[v] 탐색 {(time.perf_counter() - t0) * 1000:.0f}ms, "
          f"{reached:,}/{len(best):,}개 도달 ({reached / len(best):.0%})")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

교재와 같은 값이 나와야 합니다. 패턴 349개, 도보 환승 28,818쌍, 도달 4,156개입니다.
탐색은 교재 구현으로 80ms 안팎입니다. 한 번 돌려 정류장 4,203개 전부의 도착시각이 나옵니다.

## 6. 불변식 확인 (교재 6.7)

실제 데이터에는 손으로 아는 정답이 없습니다. 대신 반드시 성립해야 하는 성질을 확인합니다.

- 30분 늦게 출발했는데 더 일찍 도착하는 정류장이 있으면 틀린 것입니다
- 새벽 3시에 출발하면 정류장에서 첫차를 기다립니다. 도달 정류장 수는 줄지 않고, 도착시각만 늦어집니다

In [ ]:
try:
    data = sol.TransitData.from_gtfs(hanam)
    origins = data.access_stops(37.5393, 127.2148)
    early = sol.raptor(data, origins, 8 * 3600)
    later = sol.raptor(data, origins, 8 * 3600 + 1800)

    bad = sum(1 for a, b in zip(early, later) if a < INF and b < INF and b < a)
    expect("늦게 출발했는데 더 일찍 도착한 정류장", bad, 0)

    dawn = sol.raptor(data, origins, 3 * 3600)                 # 첫차 전
    misa = data.access_stops(37.5606, 127.1930)[0][0]          # 미사역에서 가장 가까운 정류장
    print(f"    새벽 3시 출발: {sum(1 for t in dawn if t < INF):,}개 도달, "
          f"{data.stop_names[misa]} 도착 {hhmm(dawn[misa])} (8시 출발이면 {hhmm(early[misa])})")
    expect("새벽 3시 출발인데 3시 전에 도착한 정류장", sum(1 for t in dawn if t < 3 * 3600), 0)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

교재 값은 새벽 3시 출발 4,195개 도달, 미사역 도착 05:14 입니다. 8시 출발이 22분 걸린 구간에 두 시간 넘게 걸립니다. 첫차를 기다리기 때문입니다.

## 7. 채점

작은 시간표 5케이스와 하남 GTFS 불변식 2개를 봅니다. `tests/test_raptor.py` 가 보는 것과 같습니다.

In [ ]:
from check import check

report = check("ch06")

## 8. 경로 복원해 보기 (교재 6.6)

`best` 는 도착시각만 알려 줍니다. 어느 버스를 어디서 탔는지는 정류장이 나아질 때마다 남긴 기록을 거꾸로 따라가야 합니다.
복원은 채점 대상이 아니라서 교재의 정돈본 `smartmob.teaching.raptor` 로 봅니다. 그쪽 `raptor` 는 리스트가 아니라 객체를 돌려주고 `result.best` 에 도착시각이 있습니다.

In [ ]:
from smartmob.teaching.raptor import TransitData, journey, raptor, summarize

ref_data = TransitData.from_gtfs(hanam)
ref_origins = ref_data.access_stops(37.5393, 127.2148)     # 하남시청
result = raptor(ref_data, ref_origins, 8 * 3600)

target = ref_data.nearest_stop(37.5606, 127.1930)          # 미사역 부근
legs = journey(ref_data, result, target)                   # 기록을 거꾸로 따라간 구간 목록
print(f"하남시청 08:00 출발 → {ref_data.stop_names[target]}")

for leg in legs:
    if leg["kind"] == "transit":
        print(f"  {leg['mode']:7s} {leg['route']:12s} "
              f"{hhmm(leg['board_time'])} → {hhmm(leg['alight_time'])}  "
              f"{leg['n_stops']}정거장 {leg['km']}km")
    else:
        print(f"  WALK    {'':12s} {leg['seconds'] / 60:.1f}분  {leg['km']}km")

summarize(ref_data, legs, 8 * 3600)

23분 걸립니다. 4장에서 같은 구간을 차로 가면 오전 첨두에 8분이었습니다.
내역을 보면 차내 6.6분, 도보 11.2분, 대기 5.1분입니다. 이동하지 않는 시간이 이동 시간보다 깁니다. 7장에서 이 내역을 통행 300건으로 넓혀 봅니다.

## 제출할 것

1. 채운 `labs/ch06_raptor.py`
2. 채점 셀의 출력 (전부 PASS)
3. 막혔던 지점과 어떻게 풀었는지 3~5줄

## 정리

- 노선이 아니라 패턴으로 묶습니다. 정류장 순서가 다르면 다른 패턴입니다. 하남은 운행 8,923개가 패턴 349개로 묶입니다
- 운행을 출발 시각 순으로 정렬해 두면 `bisect_left` 로 "이 시각 이후 첫 차"를 찾습니다
- 라운드 k 는 "k 번 탐"에 대응합니다. 먼저 내려 보고, 갈아타는 판단에는 직전 라운드 값을 씁니다
- 큰 데이터에는 정답이 없습니다. 대신 불변식으로 검증합니다
- 7장 실습에서는 여기에 환승 제한과 요금을 붙입니다